# REVE на наборе из Гвинеи-Бисау и Нигерии: эпилепсия против нормы

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-18 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Проверить REVE на втором наборе, где диагноз ставили **не по ЭЭГ**, и проверить перенос между странами (обучение в одной, проверка в другой) |
| **Данные** | Zenodo 1252141, CC BY 4.0: «EEG data collected with Emotiv device in people with epilepsy and controls in Guinea-Bissau and Nigeria». Гарнитура Emotiv EPOC, 14 каналов, 128 Гц, ~5 мин покоя (глаза открыты/закрыты). Диагноз — по истории приступов (bioRxiv 10.1101/324954), контроль — здоровые из тех же общин |
| **Модель** | REVE (`brain-bzh/reve-base`), веса **заморожены**, поверх — логистическая регрессия (C = 0,01 для всех признаков, без подбора) |
| **Деление обучение/тест** | **По людям** (несколько сессий одного человека усредняются в один вектор). Внутри страны: 5 частей × 10 повторов. Между странами: обучение в одной, проверка в другой |
| **Среда** | Kaggle, GPU T4, интернет включён |
| **Журнал** | строка в `experiments_log.md`: добавить |

Сравнение с турецким набором: `h1_2026-10-18_zs_reve-turkish.ipynb`.

## 1. Данные: скачивание (~0,43 ГБ) и таблица записей

In [1]:
import os, zipfile, urllib.request
BASE = "https://zenodo.org/records/1252141/files"
DST = "/kaggle/working/zen"
os.makedirs(DST, exist_ok=True)
for name in ["metadata_nigeria.csv", "metadata_guineabissau.csv", "EEGs_Nigeria.zip", "EEGs_Guinea-Bissau.zip"]:
    p = f"{DST}/{name}"
    if not os.path.exists(p):
        print("качаю", name)
        urllib.request.urlretrieve(f"{BASE}/{name}?download=1", p)
    if name.endswith(".zip") and not os.path.isdir(p[:-4]):
        zipfile.ZipFile(p).extractall(DST)
print(sorted(os.listdir(DST)))

качаю metadata_nigeria.csv
качаю metadata_guineabissau.csv
качаю EEGs_Nigeria.zip
качаю EEGs_Guinea-Bissau.zip
['EEGs_Guinea-Bissau', 'EEGs_Guinea-Bissau.zip', 'EEGs_Nigeria', 'EEGs_Nigeria.zip', 'metadata_guineabissau.csv', 'metadata_nigeria.csv']


In [2]:
import pandas as pd

ng = pd.read_csv(f"{DST}/metadata_nigeria.csv")
gb = pd.read_csv(f"{DST}/metadata_guineabissau.csv")
rows = []
for _, r in ng.iterrows():
    rows.append(dict(country="NG", subject=f"NG{r['subject.id']}", y=int(str(r["Group"]).lower() == "epilepsy"),
                     path=f"{DST}/EEGs_Nigeria/{r['csv.file']}"))
for _, r in gb.iterrows():
    rows.append(dict(country="GB", subject=f"GB{r['subject.id']}", y=int(str(r["Group"]).lower() == "epilepsy"),
                     path=f"{DST}/EEGs_Guinea-Bissau/signal-{r['subject.id']}.csv.gz"))
rec = pd.DataFrame(rows)

missing = rec[~rec.path.map(os.path.exists)]
print("нет файла для строк метаданных:", len(missing), missing.path.map(os.path.basename).tolist())
rec = rec[rec.path.map(os.path.exists)]
rec = rec[~rec.path.str.endswith("signal-11-1.csv.gz")]  # в метаданных: «11.1 failed, 11.2 is the right one»
assert rec.groupby("subject").y.nunique().max() == 1, "у одного человека разные метки"

print("записей:", len(rec))
print(rec.groupby(["country", "y"]).subject.nunique().rename("людей").to_frame())

нет файла для строк метаданных: 4 ['signal-11-1.csv.gz', 'signal-65-2.csv.gz', 'signal-65-1.csv.gz', 'signal-69-1.csv.gz']
записей: 318
           людей
country y       
GB      0     46
        1     51
NG      0     92
        1    128


## 2. Модель REVE и признаки

Для каждой записи:
1. Каналы гарнитуры: AF3, AF4, F3, F4, F7, F8, FC5, FC6, O1, O2, P7, P8, T7, T8 (столбцы CSV; служебные столбцы — гироскоп, качество контакта — не берём).
2. Фильтр 0,5–45 Гц по всей записи (убирает и постоянную составляющую ~4 000 мкВ), нарезка на окна по 10 с.
3. **Простые признаки:** мощности ритмов (дельта, тета, альфа, бета, гамма), абсолютные в логарифме + относительные: 14 × 5 × 2 = 140 чисел.
4. **REVE:** 128 → 200 Гц, нормировка каждого канала в окне; «по каналам» (14 × 512) и «общее среднее» (512).
5. Среднее по окнам → вектор записи; среднее по записям человека → **вектор человека**.

In [3]:
import torch
from transformers import AutoModel

pos_bank = AutoModel.from_pretrained("brain-bzh/reve-positions", trust_remote_code=True)
model = AutoModel.from_pretrained("brain-bzh/reve-base", trust_remote_code=True).eval().cuda()
print("GPU:", torch.cuda.get_device_name(0))

config.json:   0%|          | 0.00/7.72k [00:00<?, ?B/s]

configuration_bank.py:   0%|          | 0.00/277 [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/brain-bzh/reve-positions:
- configuration_bank.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


position_bank.py:   0%|          | 0.00/973 [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/brain-bzh/reve-positions:
- position_bank.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors: reconstructing file:   0%|          |  0.00B / 6.63kB            

... (8 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  277MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

GPU: Tesla T4


In [4]:
import numpy as np, scipy.signal as ss, time

FS, WIN = 128, 10
CH = ["AF3", "AF4", "F3", "F4", "F7", "F8", "FC5", "FC6", "O1", "O2", "P7", "P8", "T7", "T8"]
C = len(CH)
POS = pos_bank(CH).cuda()
print("позиции:", tuple(POS.shape))

SOS = ss.butter(4, [0.5, 45], btype="band", fs=FS, output="sos")
BANDS = [(1, 4), (4, 8), (8, 13), (13, 30), (30, 45)]

def pool(out):
    if isinstance(out, (tuple, list)): out = out[0]
    if hasattr(out, "last_hidden_state"): out = out.last_hidden_state
    B, D = out.shape[0], out.shape[-1]
    if out.dim() == 4:                              # (B, каналы, время, D)
        per_ch = out.mean(2)
    elif out.dim() == 3 and out.shape[1] % C == 0:  # (B, каналы*время, D)
        per_ch = out.reshape(B, C, -1, D).mean(2)
    else:
        raise ValueError(f"неожиданная форма выхода {tuple(out.shape)}")
    return per_ch.reshape(B, -1), out.reshape(B, -1, D).mean(1)

feats = []
t0 = time.time()
for i, r in enumerate(rec.itertuples()):
    sig = pd.read_csv(r.path, usecols=CH)[CH].to_numpy(dtype=np.float64).T   # (14, отсчёты), мкВ
    sig = ss.sosfiltfilt(SOS, sig, axis=-1)
    L = FS * WIN
    n = sig.shape[1] // L
    X = sig[:, :n * L].reshape(C, n, L).transpose(1, 0, 2)                     # (окна, 14, 1280)

    fr, P = ss.welch(X, fs=FS, nperseg=256, axis=-1)
    bp = np.stack([P[..., (fr >= a) & (fr < b)].mean(-1) for a, b in BANDS], -1)  # (окна, 14, 5)
    rel = bp / bp.sum(-1, keepdims=True)

    Xr = ss.resample_poly(X, 25, 16, axis=-1)                                   # 1280 -> 2000 отсчётов (200 Гц)
    Xr = (Xr - Xr.mean(-1, keepdims=True)) / (Xr.std(-1, keepdims=True) + 1e-8)
    ch, gl = [], []
    with torch.no_grad():
        for b in range(0, len(Xr), 32):
            x = torch.tensor(Xr[b:b + 32], dtype=torch.float32).cuda()
            out = model(x, POS.expand(x.size(0), -1, -1))
            if i == 0 and b == 0:
                o = out[0] if isinstance(out, (tuple, list)) else getattr(out, "last_hidden_state", out)
                print("форма выхода REVE:", tuple(o.shape))
            a1, a2 = pool(out); ch.append(a1.cpu()); gl.append(a2.cpu())

    feats.append(dict(
        subject=r.subject, country=r.country, y=r.y, n_win=n,
        amp=np.median(X.std(-1)),
        bp=np.concatenate([np.log(bp).mean(0).ravel(), rel.mean(0).ravel()]),
        gl=torch.cat(gl).mean(0).numpy(), ch=torch.cat(ch).mean(0).numpy(),
    ))
    if (i + 1) % 25 == 0 or i + 1 == len(rec):
        print(f"{i+1}/{len(rec)} {time.time()-t0:.0f} с")

# среднее по сессиям одного человека
df = pd.DataFrame(feats)
subj = df.groupby("subject").agg(country=("country", "first"), y=("y", "first"),
                                 n_win=("n_win", "sum"), amp=("amp", "mean")).reset_index()
F = {k: np.stack([np.mean(np.stack(df.loc[df.subject == s, k].tolist()), 0) for s in subj.subject])
     for k in ["bp", "gl", "ch"]}
F["amp"] = subj["amp"].to_numpy()[:, None]
F["rel"] = F["bp"][:, F["bp"].shape[1] // 2:]   # только относительные мощности (без амплитуды)
y, country = subj["y"].to_numpy(), subj["country"].to_numpy()
np.savez("/kaggle/working/feats_africa.npz", y=y, country=country, subject=subj.subject.to_numpy(), **F)
print("людей:", len(subj), "| эпилепсия:", int(y.sum()), "| контроль:", int((1 - y).sum()))

позиции: (14, 3)
форма выхода REVE: (27, 14, 11, 512)
25/318 9 с
50/318 17 с
75/318 25 с
100/318 33 с
125/318 40 с
150/318 49 с
175/318 56 с
200/318 65 с
225/318 73 с
250/318 83 с
275/318 94 с
300/318 105 с
318/318 112 с
людей: 317 | эпилепсия: 179 | контроль: 138


## 3. Проверка на перекосы

Длина записи и уровень сигнала по группам в каждой стране. Если группы сильно различаются по амплитуде, высокие оценки могут объясняться не эпилепсией.

In [5]:
for c in ["NG", "GB"]:
    m = country == c
    print(f"{c}: окон (медиана) эпилепсия {np.median(subj.n_win[m & (y == 1)]):.0f}, контроль {np.median(subj.n_win[m & (y == 0)]):.0f} | "
          f"амплитуда эпилепсия {np.median(F['amp'][m & (y == 1)]):.1f}, контроль {np.median(F['amp'][m & (y == 0)]):.1f} мкВ")

NG: окон (медиана) эпилепсия 26, контроль 26 | амплитуда эпилепсия 18.4, контроль 19.3 мкВ
GB: окон (медиана) эпилепсия 30, контроль 30 | амплитуда эпилепсия 19.4, контроль 17.2 мкВ


## 4. Модель и метрики

Логистическая регрессия, **C = 0,01 для всех признаков** (без подбора — чтобы не завышать). ROC AUC на уровне человека.

**4.1. Внутри страны** (5 частей × 10 повторов).

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

NAMES = {"amp": "Только амплитуда", "rel": "Относительные мощности", "bp": "Мощности ритмов (все)",
         "gl": "REVE, общее среднее", "ch": "REVE, по каналам"}
clf = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=5000))
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)

for c in ["NG", "GB", "ALL"]:
    m = np.ones_like(y, bool) if c == "ALL" else country == c
    print(f"\n=== {c}: {m.sum()} человек (эпилепсия {y[m].sum()}, контроль {(1 - y[m]).sum()})")
    for k, name in NAMES.items():
        s = cross_val_score(clf(), F[k][m], y[m], cv=cv, scoring="roc_auc")
        print(f"  {name:24s} ROC AUC = {s.mean():.3f} ± {s.std():.3f}")


=== NG: 220 человек (эпилепсия 128, контроль 92)
  Только амплитуда         ROC AUC = 0.456 ± 0.069
  Относительные мощности   ROC AUC = 0.892 ± 0.041
  Мощности ритмов (все)    ROC AUC = 0.918 ± 0.041
  REVE, общее среднее      ROC AUC = 0.886 ± 0.048
  REVE, по каналам         ROC AUC = 0.959 ± 0.030

=== GB: 97 человек (эпилепсия 51, контроль 46)
  Только амплитуда         ROC AUC = 0.670 ± 0.103
  Относительные мощности   ROC AUC = 0.820 ± 0.090
  Мощности ритмов (все)    ROC AUC = 0.865 ± 0.087
  REVE, общее среднее      ROC AUC = 0.919 ± 0.049
  REVE, по каналам         ROC AUC = 0.925 ± 0.049

=== ALL: 317 человек (эпилепсия 179, контроль 138)
  Только амплитуда         ROC AUC = 0.517 ± 0.072
  Относительные мощности   ROC AUC = 0.823 ± 0.036
  Мощности ритмов (все)    ROC AUC = 0.892 ± 0.032
  REVE, общее среднее      ROC AUC = 0.862 ± 0.037
  REVE, по каналам         ROC AUC = 0.932 ± 0.027


**4.2. Перенос между странами:** обучение в одной стране, проверка в другой. 95% доверительный интервал — бутстреп по людям тестовой страны.

In [7]:
def boot_ci(yt, p, n=2000, seed=0):
    rng = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = rng.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], p[i]))
    return np.percentile(v, [2.5, 97.5])

for tr, te in [("NG", "GB"), ("GB", "NG")]:
    a, b = country == tr, country == te
    print(f"\n=== обучение {tr} ({a.sum()} чел.) -> проверка {te} ({b.sum()} чел.)")
    for k, name in NAMES.items():
        p = clf().fit(F[k][a], y[a]).predict_proba(F[k][b])[:, 1]
        lo, hi = boot_ci(y[b], p)
        print(f"  {name:24s} ROC AUC = {roc_auc_score(y[b], p):.3f}  (95% ДИ {lo:.2f}–{hi:.2f})")


=== обучение NG (220 чел.) -> проверка GB (97 чел.)
  Только амплитуда         ROC AUC = 0.671  (95% ДИ 0.56–0.78)
  Относительные мощности   ROC AUC = 0.596  (95% ДИ 0.48–0.71)
  Мощности ритмов (все)    ROC AUC = 0.671  (95% ДИ 0.56–0.78)
  REVE, общее среднее      ROC AUC = 0.417  (95% ДИ 0.30–0.54)
  REVE, по каналам         ROC AUC = 0.516  (95% ДИ 0.40–0.64)

=== обучение GB (97 чел.) -> проверка NG (220 чел.)
  Только амплитуда         ROC AUC = 0.462  (95% ДИ 0.39–0.54)
  Относительные мощности   ROC AUC = 0.704  (95% ДИ 0.64–0.77)
  Мощности ритмов (все)    ROC AUC = 0.754  (95% ДИ 0.69–0.82)
  REVE, общее среднее      ROC AUC = 0.683  (95% ДИ 0.61–0.75)
  REVE, по каналам         ROC AUC = 0.603  (95% ДИ 0.52–0.68)


### 4.3. Проверка: когда записывали группы

Перенос между странами провалился при высоких оценках внутри страны. Проверяем по метаданным, записывали ли больных и контроль в одни и те же дни.

In [ ]:
for f in ["metadata_nigeria.csv", "metadata_guineabissau.csv"]:
    d = pd.read_csv(f"{DST}/{f}")
    d["день"] = pd.to_datetime(d.startTime, dayfirst=True).dt.date
    d["группа"] = d.Group.str.lower()
    print(f); print(pd.crosstab(d["день"], d["группа"])); print()

metadata_nigeria.csv
группа      control  epilepsy
день
2016-09-26        7         0
2016-09-27        1         0
2016-09-28        0        69
2016-09-29        9         0
2016-09-30       53         0
2016-10-01        0        59
2016-10-03       27         0

metadata_guineabissau.csv
группа      control  epilepsy
день
2020-05-23        5         0
2020-05-24        0         2
2020-05-26        3         0
2020-05-27        3        48
2020-05-28        8         0
2020-05-29       13         0
2020-05-31       14         1

## 5. Выводы

| Признаки | Нигерия (внутри) | Гвинея-Бисау (внутри) | NG → GB | GB → NG |
|---|---|---|---|---|
| Только амплитуда | 0,46 | 0,67 | 0,67 | 0,46 |
| Относительные мощности | 0,89 | 0,82 | 0,60 | 0,70 |
| Мощности ритмов (все) | 0,92 | 0,87 | **0,67** | **0,75** |
| REVE, общее среднее | 0,89 | 0,92 | 0,42 | 0,68 |
| REVE, по каналам | **0,96** | **0,93** | 0,52 | 0,60 |

1. **Внутри страны оценки очень высокие** (REVE по каналам 0,93–0,96), и амплитуда их не объясняет (0,46–0,67).
2. **Но больных и контроль записывали в разные дни** (раздел 4.3): в Нигерии нет ни одного дня, где были бы обе группы; в Гвинее-Бисау 48 из 51 больного записаны 27.05. Модель может различать **условия записи** (место, шум вокруг, контакт электродов, состояние гарнитуры), а не болезнь. Поэтому оценки «внутри страны» доверия не заслуживают.
3. **Честная проверка — перенос между странами.** Тут картина обратная: простые мощности ритмов держатся лучше всех (0,67 и 0,75), а REVE по каналам падает почти до угадывания (0,52 и 0,60). Замороженные признаки REVE (7 168 чисел на 97–220 человек) сильнее всего подхватывают условия записи и хуже всего переносятся.
4. **Для турецкого набора** (REVE 0,94) даты записи неизвестны, поэтому такой же перекос там исключить нельзя — та оценка, скорее всего, тоже завышена.

**Главный урок:** на открытых наборах, где группы собраны отдельно, высокая оценка внутри набора ничего не доказывает. Каждую модель нужно проверять переносом на другой набор/место/день.

**Ограничения:** гарнитура потребительская (14 каналов, 128 Гц), запись 5 минут, контроль — здоровые; возраст в метаданных отсутствует; REVE не дообучалась; C не подбиралось (0,01 для всех).

## 6. Что дальше

- В TUEP проверить, не собраны ли группы в разные периоды/на разных аппаратах; оценивать с учётом этого.
- Для REVE: меньше признаков (сжатие, усреднение по областям) или дообучение — и смотреть именно на перенос.
- Простые мощности ритмов оставить как обязательный эталон в каждом эксперименте.